# 🔗 Multi-Hop RAG — Chain Your Searches

## What Is Multi-Hop?

Some questions require **connecting information from multiple documents**.

```
Question: "What language does the tool that Alice recommended use?"

Hop 1: What tool did Alice recommend?  →  "Alice recommended FastAPI"
Hop 2: What language does FastAPI use?  →  "FastAPI is built with Python"

Answer: Python
```

One search can't answer this — you need to **hop** through the knowledge base.

## The Pattern

```
Search 1  →  Read result  →  Extract new search term  →  Search 2  →  Answer
```

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Knowledge base where information is spread across docs
docs = [
    "Alice is the backend lead at TechCorp.",
    "Alice recommended using FastAPI for the new API service.",
    "FastAPI is a modern web framework for building APIs with Python.",
    "FastAPI supports async by default and has automatic docs generation.",
    "Python is known for its clean syntax and large ecosystem.",
    "Bob is the frontend lead and recommended React for the UI.",
    "React is a JavaScript library for building user interfaces.",
]

doc_embeddings = model.encode(docs)

def search(query, top_k=2):
    q_emb = model.encode(query)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in top_idx]

print("Ready!")

In [ ]:
# Single-hop fails on this question

question = "What language does the tool Alice recommended use?"

single_results = search(question)
print("Single-hop search:")
for r in single_results:
    print(f"  → {r}")

print()
print("Problem: We found 'Alice recommended FastAPI' but not what language FastAPI uses.")
print("We need a SECOND search using 'FastAPI' as the new query.")

In [ ]:
# Multi-hop search — two hops

print(f"Question: {question}")
print("=" * 55)

# Hop 1: search for Alice's recommendation
hop1_query = "what tool did Alice recommend"
hop1_results = search(hop1_query)
print(f"\nHop 1 — '{hop1_query}'")
for r in hop1_results:
    print(f"  → {r}")

# Extract the key entity from hop 1 (would be done by LLM in production)
# We manually extract "FastAPI" here
extracted_entity = "FastAPI"
print(f"\nExtracted entity: '{extracted_entity}'")

# Hop 2: search for information about FastAPI
hop2_query = f"{extracted_entity} programming language"
hop2_results = search(hop2_query)
print(f"\nHop 2 — '{hop2_query}'")
for r in hop2_results:
    print(f"  → {r}")

# Combine context from both hops
all_context = hop1_results + hop2_results
print(f"\nCombined context ({len(all_context)} docs):")
for c in all_context:
    print(f"  - {c}")

print("\nFinal Answer: FastAPI uses Python.")

In [ ]:
# Generic multi-hop pipeline

def multi_hop_search(question, hops=2):
    """
    Performs multi-hop retrieval:
    - Each hop refines the search based on previous results
    - In production, an LLM generates each hop's query
    """
    all_docs = []
    current_query = question

    for hop_num in range(1, hops + 1):
        results = search(current_query, top_k=2)
        print(f"Hop {hop_num} — '{current_query}'")
        for r in results:
            print(f"  → {r}")
        all_docs.extend(results)

        # In production: LLM reads results and writes next query
        # Here: we append key terms from results to refine the next search
        if hop_num < hops:
            # Crude entity extraction — just take first unique nouns
            new_terms = []
            for r in results:
                words = r.split()
                for w in words:
                    if w[0].isupper() and w not in current_query:
                        new_terms.append(w)
                        break
            if new_terms:
                current_query = " ".join(new_terms[:2]) + " language technology"
        print()

    return list(dict.fromkeys(all_docs))  # Deduplicate, preserve order


print("Multi-hop for: 'What language does the tool Alice recommended use?'")
print("=" * 60)
context = multi_hop_search("What language does the tool Alice recommended use?", hops=2)
print(f"Total unique docs collected: {len(context)}")

## Key Takeaways 🎯

- **Multi-hop** is needed when answering requires connecting facts from multiple docs
- Each hop uses results from the previous hop to form a new, more targeted query
- In production: an LLM reads each hop's results and writes the next query
- **Signals you need multi-hop:** questions with "who made", "what does X use", "where did Y go"

```
1 hop  → simple factual questions
2 hops → entity resolution ("what X uses Y")
3+ hops → complex reasoning chains
```